In [1]:
import os
# os.environ["MKL_NUM_THREADS"] = "1"
os.environ["NUMEXPR_MAX_THREADS"] = "128"  # Or whatever upper limit you want  
import sys
sys.path.append('../')
from datetime import datetime
import pytz
import numpy as np
import scqubits as scq
import scqubits.settings as settings
import qutip as qt
from joblib import Parallel, delayed
import pandas as pd
import utils_2Q_gate_zp as ut
settings.OVERLAP_THRESHOLD = 0.3  # Update scqubits settings

In [2]:
drive_phi, drive_theta, n_full = False, True, 150
# drive_phi, drive_theta, n_full = True, False, 500 # 50 states →12 workers, (100 states/40 workers, 200/160). 
t1 = 170 # μs
tg_list = [9] # [1, 5, 9, 13, 17 ] # np.arange(18).tolist() #  ## 18 for theta, 19 for phi     
charge_truc = True  # whether to truncate the charge space
calculate_ideal, calculate_noise = False, True # True, False # False, True #   whether to calculate noisy fidelity
num_cpus = 4 # Lower num_cpus <4 can reduce num of workers while >4 won’t change the num.
apply_decay, apply_dephase = True, True # False, True # True, False #

max_step_ideal, max_step_noisy = 1e-3, 3e-4 # Set max_step to 0 for parallel execution
print("drive_phi=", drive_phi, "; drive_theta =", drive_theta, "; n_full =", n_full, "; charge_truc =", charge_truc)
print(f"T1 = Tphi = {t1} μs, num_cpus = {num_cpus}")
print(f"calculate_ideal = {calculate_ideal}, calculate_noise = {calculate_noise}")
option_ideal, option_noisy = ut.get_qutip_options(max_step_ideal, max_step_noisy) 
print(f'apply_decay = {apply_decay}, apply_dephase = {apply_dephase}')

gamma_t1 = 1 / 1e3 / t1 # calculate decay rate given T1, unit in micro-second
tphi = t1 # calculate decay rate given T1, unit in micro-second

# Load data
evals, n_theta, n_phi, logi_state = ut.load_qubit_data_xgate() # Load spectrum and matrix elements
params = ut.load_drive_params_xgate(drive_theta)[tg_list, ]  # [1::4,] # Load pulse parameters from CSV
n_job = len(params)    

# Build Hamiltonian
w_trans_1, w_trans_2, drive_term, Gamma_t1 = ut.compute_drive_xgate(evals, n_theta, n_phi, drive_phi, drive_theta, gamma_t1)  
hspace = np.arange(n_full).tolist() # do not truncate
if charge_truc:
    hspace = ut.get_truncated_subspace_xgate(drive_term, n_full)
values_to_remove = {} # remove elements that give nan
hspace = [item for item in hspace if item not in values_to_remove]

n_hspace = len(hspace)
H_qbt_drive, drive_truc, logi_idx = ut.build_hamiltonian_xgate(evals, drive_term, hspace, logi_state)

# Print summary
print("n_hspace =", n_hspace, ";   n_job = ", n_job)
ut.print_data(f'hspace ({n_full}\{n_hspace})', hspace, num_each_row=10)
ut.print_data(f'params', params.tolist(), num_each_row=1)


drive_phi= False ; drive_theta = True ; n_full = 150 ; charge_truc = True
T1 = Tphi = 170 μs, num_cpus = 4
calculate_ideal = False, calculate_noise = True
Ideal: max_step = 0.001, nsteps = 1000.0
Noisy: max_step = 0.0003, nsteps = 3333.3333333333335
apply_decay = True, apply_dephase = True


n_hspace = 78 ;   n_job =  1

hspace (150\78) = np.array([
0, 1, 2, 4, 5, 7, 8, 11, 12, 16 ,
17, 18, 21, 23, 25, 27, 30, 31, 32, 33 ,
37, 38, 40, 41, 42, 45, 46, 47, 48, 51 ,
52, 56, 59, 62, 63, 64, 65, 67, 72, 73 ,
74, 76, 78, 79, 82, 83, 85, 87, 88, 90 ,
92, 93, 96, 97, 98, 102, 103, 106, 107, 112 ,
114, 118, 119, 120, 121, 122, 123, 124, 127, 128 ,
131, 132, 133, 134, 137, 138, 142, 143 ,
])

params = np.array([
[55.267933, 0.073988, 0.020007, 0.001756, 0.002668] ,
])


In [9]:
# Load data and prepare operators for noisy fidelity simulation
state_idx_tphi, gamma_dephase_new = ut.load_dephasing_data_xgate(drive_theta) # Load dephasing data
c_op_list = ut.construct_c_ops_xgate(n_hspace, drive_truc, Gamma_t1, gamma_dephase_new, 
                            tphi, hspace, state_idx_tphi, apply_decay, apply_dephase) # Construct collapse operators

len_t1 = 3003
max_value = []
for i,_ in enumerate(c_op_list):
    max_value.append( abs(np.max(c_op_list[i].full())))
    # print(f'c_ops[{i}] = {abs(  np.max(c_op_list[i].full()) ):.8f}') # for debugging
print(f'decay_max = {np.max(max_value[:len_t1]):.8f}, dephase_max = {np.max(max_value[len_t1:]):.8f}' )
print(f'decay_max_idx = {np.argmax(max_value[:len_t1])}, dephase_max = {np.argmax(max_value[len_t1:])}' )

np.shape(jump_t1)= (3003, 78, 78) ; np.shape(jump_tphi)= (78, 78, 78)
decay_max = 0.00883527, dephase_max = 0.00428188
decay_max_idx = 2555, dephase_max = 13


decay_max = 0.00883527, dephase_max = 0.00428188
decay_max_idx = 2555, dephase_max = 13


In [6]:
78*77/2

3003.0

In [4]:
print(f'np.shape(c_op_list) = {np.shape(c_op_list)}')     


np.shape(c_op_list) = (3081, 78, 78)


In [5]:
=

SyntaxError: invalid syntax (1763773627.py, line 1)

In [ ]:
# Noisy fidelity simulation
args = [H_qbt_drive, w_trans_1, w_trans_2, num_cpus, c_op_list, logi_idx, option_ideal, option_noisy]
f_noise = Parallel(n_jobs=n_job)(delayed(ut.xgate_fidelity_log_noise)(args_indep, *args) for args_indep in params)
ut.print_data(f'f_{t1}us_{n_hspace}', f_noise)